In [15]:
pip install python-dotenv langchain-classic

16054.43s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [23]:
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [24]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [25]:
import sys
sys.path.append("../backend")

from app.core.schemas import IncidentDiagnosis, ExtractedEntities
from app.core.parsers import(
    get_structured_response,
    get_fallback_structured_response,
    get_structured_response_with_fallback,
)

In [26]:
sample_alerts = [
    "checkout-service p99 latency +400% for 5 minutes, logs show 'connection pool exhausted (redis-pool-1)'",
    "api-gateway returning NET::ERR_CERT_DATE_INVALID to all external clients since 03:14 UTC",
    "notification-service pods OOMKilled repeatedly, memory climbing steadily over 6-8hr windows",
]

for alert in sample_alerts:
    result = get_structured_response(llm, IncidentDiagnosis, alert)
    print(f"\nAlert: {alert[:60]}...") # basically way to slice string, alert jitta bhi bda genrate hoga uska bs 60 words/char hi chahiye hme
    print("Type:", type(result))
    print("Parsed:", result.model_dump()) #model_dump is used to convert pydantic into dictonary format
    assert isinstance(result, IncidentDiagnosis) # assert is used to validate whether first parameter satisfy the 2nd one, or like it exist, means value is true or not, here result must be of IncidentDiagnosis type, otherwise will not accept the result


Alert: checkout-service p99 latency +400% for 5 minutes, logs show ...
Type: <class 'app.core.schemas.IncidentDiagnosis'>
Parsed: {'severity': <IncidentSeverity.HIGH: 'high'>, 'root_cause': 'connection pool exhaustion', 'affected_services': ['checkout-service', 'redis-pool-1'], 'confidence_score': 0.8}

Alert: api-gateway returning NET::ERR_CERT_DATE_INVALID to all exte...
Type: <class 'app.core.schemas.IncidentDiagnosis'>
Parsed: {'severity': <IncidentSeverity.HIGH: 'high'>, 'root_cause': 'Expired or invalid SSL certificate', 'affected_services': ['api-gateway'], 'confidence_score': 0.9}

Alert: notification-service pods OOMKilled repeatedly, memory climb...
Type: <class 'app.core.schemas.IncidentDiagnosis'>
Parsed: {'severity': <IncidentSeverity.HIGH: 'high'>, 'root_cause': 'Memory leak in notification-service pods', 'affected_services': ['notification-service'], 'confidence_score': 0.8}


In [27]:
incident_note = """
Reported by Priya Nair on March 14th around 2:30am. Escalated to Marcus Chen.
Affected: checkout-service, redis-pool-1, and the payment-validator sidecar.
Second occurrence since the Feb 2nd deploy.
"""

entities = get_structured_response(llm, ExtractedEntities, incident_note)
print(entities.model_dump()) #model_dump use to convert pydantic schema into dict.
assert "Priya Nair" in entities.names or any("Priya" in n for n in entities.names)
# again assert here will check whether Priya Nair exist in name of entites or not, if not atleast a first word priya exist in entities name or not. so that bcz of word restriction we don't loose the entities and data, like maybe sometime llm instead of giving 
# full name can write first name only. so as a result we don't want to reject our result bcz of that


{'names': ['Priya Nair', 'Marcus Chen'], 'dates': ['March 14th', '2:30am', 'Feb 2nd'], 'system_components': ['checkout-service', 'redis-pool-1', 'payment-validator']}


In [28]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser

malformed_output = """
{
  "severity": "extremely bad",
  "root_cause": "redis pool exhausted",
  "affected_services": "checkout-service",
  "confidence_score": "high"
}
"""
# Three separate validation failures at once:
#   - "extremely bad" is not a valid IncidentSeverity enum value
#   - affected_services is a string, schema requires List[str]
#   - confidence_score is a string, schema requires float in [0,1]

base_parser = PydanticOutputParser(pydantic_object=IncidentDiagnosis)

print("--- Attempting strict parse (expected to fail) ---")
try:
    base_parser.parse(malformed_output)
except Exception as e:
    print ("Validation failed as expected:\n", e)


print("\n--- Repairing with OutputFixingParser ---")
fixing_parser = OutputFixingParser.from_llm(parser=base_parser, llm = llm)
fixed = fixing_parser.parse(malformed_output)
print("Repaired object:", fixed.model_dump())
assert isinstance(fixed, IncidentDiagnosis)

--- Attempting strict parse (expected to fail) ---
Validation failed as expected:
 Failed to parse IncidentDiagnosis from completion {"severity": "extremely bad", "root_cause": "redis pool exhausted", "affected_services": "checkout-service", "confidence_score": "high"}. Got: 3 validation errors for IncidentDiagnosis
severity
  Input should be 'low', 'medium', 'high' or 'critical' [type=enum, input_value='extremely bad', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum
affected_services
  Input should be a valid list [type=list_type, input_value='checkout-service', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/list_type
confidence_score
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='high', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing
For troubleshooting, visit: https://docs.langchain.com/oss/python/l

In [29]:
import time

test_input = sample_alerts[0]

start = time.perf_counter()
get_structured_response(llm, IncidentDiagnosis, test_input)
primary_time = time.perf_counter() - start

start = time.perf_counter()
get_fallback_structured_response(llm, IncidentDiagnosis, test_input)
fallback_time = time.perf_counter() - start

print(f"Primary path:  {primary_time:.2f}s")
print(f"Fallback path: {fallback_time:.2f}s  (1 call, or 2 if repair triggers)")

Primary path:  0.53s
Fallback path: 1.87s  (1 call, or 2 if repair triggers)
